In [ ]:
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# ==========================================
# 1. 데이터 로드 및 분할
# ==========================================
print("1. 데이터 로드 중...")
X_raw = np.load('X32_2.npy')
y = np.load('y32_2.npy')
print(f"   - 전체 데이터 형태: X={X_raw.shape}, y={y.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# ==========================================
# 2. 스케일링 (Data Leakage 방지)
# ==========================================
print("2. 스케일링 진행 중...")
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 3. 극한의 피처 다이어트 (Selector)
# ==========================================
print("\n3. 피처 핵심 특징 추출 중...")
# 임시 모델로 피처 중요도 파악
base_rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
base_rf.fit(X_train_scaled, y_train)

# 환경 볼륨에 영향받지 않는 핵심 피처 20개만 남김
selector = SelectFromModel(base_rf, prefit=True, max_features=20, threshold=-np.inf)

X_train_selected = selector.transform(X_train_scaled)
X_test_selected = selector.transform(X_test_scaled)
print(f"원래 {X_train_scaled.shape[1]}개 -> 핵심 {X_train_selected.shape[1]}개 피처로 압축됨!")

# ==========================================
# 4. 예민하게 세팅
# ==========================================
print("\n4. 학습 시작...")
rf_model_regularized = RandomForestClassifier(
    n_estimators=100,
    max_depth=7,                 # 복잡한 공격도 잡아내기 위한 지능
    min_samples_split=10,        
    min_samples_leaf=4,          
    max_features='sqrt',
    class_weight={0: 1, 1: 10},  # ★ 핵심 1: 악성(1)을 놓치는 것에 10배의 수학적 벌점 부과
    random_state=42,
    n_jobs=-1
)

# 다이어트된 20개 피처로 최종 모델 학습
rf_model_regularized.fit(X_train_selected, y_train)

# ==========================================
# 5.Threshold 극단적 하향
# ==========================================
THRESHOLD = 0.15 
print(f"\n[현재 임계값: {THRESHOLD}] (단 {THRESHOLD*100}%만 의심스러워도 즉각 차단)")

# 예측 확률 뽑기 (압축된 테스트 데이터 사용)
train_pred_prob = rf_model_regularized.predict_proba(X_train_selected)[:, 1]
test_pred_prob = rf_model_regularized.predict_proba(X_test_selected)[:, 1]

# 임계값 적용하여 0과 1로 변환
train_pred_strict = (train_pred_prob >= THRESHOLD).astype(int)
test_pred_strict = (test_pred_prob >= THRESHOLD).astype(int)

# ==========================================
# 6. 최종 성능 평가 (과적합 진단 및 지표 확인)
# ==========================================
train_f1 = f1_score(y_train, train_pred_strict, average='weighted')
test_f1 = f1_score(y_test, test_pred_strict, average='weighted')

print("\n[과적합 진단 결과]")
print(f"- 훈련 데이터 F1-Score : {train_f1:.6f}")
print(f"- 테스트 데이터 F1-Score: {test_f1:.6f}")
print(f"- 점수 격차(Gap)       : {abs(train_f1 - test_f1):.6f}")

acc = accuracy_score(y_test, test_pred_strict)
prec = precision_score(y_test, test_pred_strict, average='binary', zero_division=0)
rec = recall_score(y_test, test_pred_strict, average='binary', zero_division=0) 

cm = confusion_matrix(y_test, test_pred_strict)
tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

print("\n==================================================")
print("[최종 방어 모델 성능]")
print(f"- Accuracy (정확도) : {acc:.4f}")
print(f"- Precision(정밀도) : {prec:.4f} (정상 오탐으로 인해 낮아질 수 있음)")
print(f"- Recall   (탐지율) : {rec:.4f} (★ 가장 중요: 공격을 얼마나 잘 잡아냈는가!)")
print(f"- F1-Score          : {test_f1:.4f}")
print("--------------------------------------------------")
print(f"오탐(False Positive, 정상인데 차단됨): {fp}건 (방어망을 좁혀서 증가함)")
print(f"미탐(False Negative, 공격인데 뚫림)  : {fn}건 (★ 이 수치가 극도로 낮아야 함!)")
print(f"오탐률(FPR)      : {fpr*100:.4f}%")
print("==================================================")

# ==========================================
# 7. 백엔드 배포용 파일 저장 (3종 세트)
# ==========================================
print("\n7. 배포용 파일 생성 중...")
joblib.dump(rf_model_regularized, 'ids_rf_model_sensitive.joblib')
joblib.dump(scaler, 'ids_robust_scaler_1.joblib')
joblib.dump(selector, 'ids_feature_selector_1.joblib') 

print("✅ [완료] 모델(model), 스케일러(scaler), 거름망(selector) 3종 파일 저장 성공!")

1. 데이터 로드 중...
   - 전체 데이터 형태: X=(200000, 62), y=(200000,)
2. 스케일링 진행 중...

3. [피처 다이어트] 핵심 특징 추출 중...
👉 다이어트 완료: 원래 62개 -> 핵심 20개 피처로 압축됨!

4. [초예민 모델] 학습 시작...

💡 [현재 임계값: 0.15] (단 15.0%만 의심스러워도 즉각 차단!)

[과적합 진단 결과]
- 훈련 데이터 F1-Score : 0.941545
- 테스트 데이터 F1-Score: 0.942587
- 점수 격차(Gap)       : 0.001042

[🔥 최종 방어 모델 성능 (초예민 모드)]
- Accuracy (정확도) : 0.9428
- Precision(정밀도) : 0.8973 (정상 오탐으로 인해 낮아질 수 있음)
- Recall   (탐지율) : 1.0000 (★ 가장 중요: 공격을 얼마나 잘 잡아냈는가!)
- F1-Score          : 0.9426
--------------------------------------------------
🚨 오탐(False Positive, 정상인데 차단됨): 2289건 (방어망을 좁혀서 증가함)
🚨 미탐(False Negative, 공격인데 뚫림)  : 0건 (★ 이 수치가 극도로 낮아야 함!)
🚨 오탐률(FPR)      : 11.4450%

7. 배포용 파일 생성 중...
✅ [완료] 모델(model), 스케일러(scaler), 거름망(selector) 3종 파일 저장 성공!
